In [1]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_URL = "https://pro.api.openmeasures.io"
jwt_token = os.getenv("OPEN_MEASURES_TOKEN")

_headers = {
    "Authorization": f"Bearer {jwt_token}"
}

In [4]:
# see quota
response = requests.get(f"{API_URL}/quota", headers=_headers)
print(response.status_code)
print(response.json())

200
{'organization_usage': {'current_year_month': '2026-06', 'last_active': '2026-06-15T19:14:09.573301Z', 'core_api_monthly_requests_count': 7, 'core_api_monthly_media_requests_count': 0, 'core_api_monthly_thumbnail_media_requests_count': 156, 'core_api_monthly_ai_moderation_requests_count': 0, 'core_api_monthly_entity_recognition_requests_count': 0, 'network_graph_monthly_requests_count': 0, 'network_graphs_count': 0, 'search_app_monthly_requests_count': 0, 'crawl_requests_usage': {'keyword': 0, 'profile': 0, 'telegram': 0, 'whatsapp': 0, 'channel': 0}, 'ai_credits_percentage': 0.0}, 'monthly_limits': {'core_api_global_access': False, 'core_api_global_media_access': False, 'core_api_monthly_request_limit': 10000, 'core_api_monthly_media_request_limit': 1000, 'core_api_monthly_thumbnail_media_request_limit': -1, 'core_api_monthly_ai_moderation_request_limit': 0, 'core_api_monthly_entity_recognition_request_limit': 0, 'core_api_time_cutoff_days': -1, 'core_api_time_until_cutoff_days': 

In [12]:
# Check and compare query sizes

explicit = '"jew*" OR "joo" OR "joos" OR "kike*" OR "ZOG" OR "holocaust" OR ("(((" AND ")))") OR "the JQ" OR "globalist" OR "soros" OR "rothschild" OR "zioni*" OR "zio" OR "zios" OR "mossad" OR "aipac"'

queries = {
    "israel_not": f'("israel*" NOT ({explicit}))',
    "israel_and": f'("israel*" AND ({explicit}))',
    "no_israel": f'({explicit})',
}

sites_to_test = ["4chan", "8kun", "truthsocial", "bluesky"]

results_summary = {}

for query_label, term_query in queries.items():
    results_summary[query_label] = {}
    for site in sites_to_test:
        test_params = {
            "sortdesc": "true",
            "limit": 1,
            "site": site,
            "term": term_query,
            "since": "2025-01-01",
            "until": "2025-12-31",
            "standard_fields": "true",
            "querytype": "boolean_content",
        }
        response = requests.get(f"{API_URL}/content", headers=_headers, params=test_params)
        print(f"====| {query_label} | {site} |====")
        print(f"Status: {response.status_code}")
        if response.status_code == 200:
            data = response.json()
            total_hits = data.get('total_hits')
            print(f"Total hits: {total_hits}")
            results_summary[query_label][site] = total_hits
        else:
            print(response.text[:300])
            results_summary[query_label][site] = None
        print()

print("\n=== SUMMARY ===")
for query_label, site_hits in results_summary.items():
    print(f"\n{query_label}:")
    for site, hits in site_hits.items():
        print(f"  {site}: {hits}")

====| israel_not | 4chan |====
Status: 200
Total hits: 598933

====| israel_not | 8kun |====
Status: 200
Total hits: 22338

====| israel_not | truthsocial |====
Status: 200
Total hits: 651488

====| israel_not | bluesky |====
Status: 200
Total hits: 2639162

====| israel_and | 4chan |====
Status: 200
Total hits: 73148

====| israel_and | 8kun |====
Status: 200
Total hits: 4765

====| israel_and | truthsocial |====
Status: 200
Total hits: 50678

====| israel_and | bluesky |====
Status: 200
Total hits: 58174

====| no_israel | 4chan |====
Status: 200
Total hits: 1228316

====| no_israel | 8kun |====
Status: 200
Total hits: 28755

====| no_israel | truthsocial |====
Status: 200
Total hits: 580635

====| no_israel | bluesky |====
Status: 200
Total hits: 664819


=== SUMMARY ===

israel_not:
  4chan: 598933
  8kun: 22338
  truthsocial: 651488
  bluesky: 2639162

israel_and:
  4chan: 73148
  8kun: 4765
  truthsocial: 50678
  bluesky: 58174

no_israel:
  4chan: 1228316
  8kun: 28755
  truthso

In [13]:
import pandas as pd

# results_summary structure: {query_label: {site: total_hits}}
df = pd.DataFrame(results_summary).T  # rows = query variant, columns = site
df.index.name = "query_variant"

# reconstruct 'full' as the sum of the three partitions (sanity check + denominator)
df.loc["full"] = df.loc["israel_not"] + df.loc["israel_and"] + df.loc["no_israel"]

print("=== Raw hit counts per stratum ===")
print(df.to_string())

print("\n=== Composition: % of full corpus each stratum represents ===")
composition_df = (df.div(df.loc["full"], axis=1) * 100).round(2)
print(composition_df.to_string())

print("\n=== Composition by site ===")
for site in df.columns:
    print(f"\n{site}: total = {df.loc['full', site]:,}")
    for stratum in ["no_israel", "israel_and", "israel_not"]:
        pct = composition_df.loc[stratum, site]
        count = df.loc[stratum, site]
        print(f"  {stratum:<12} {count:>10,} ({pct:>5.2f}%)")

print("\n=== Cross-platform share: where does each stratum's volume come from? ===")
row_share_df = (df.div(df.sum(axis=1), axis=0) * 100).round(2)
print(row_share_df.loc[["no_israel", "israel_and", "israel_not"]].to_string())

=== Raw hit counts per stratum ===
                 4chan   8kun  truthsocial  bluesky
query_variant                                      
israel_not      598933  22338       651488  2639162
israel_and       73148   4765        50678    58174
no_israel      1228316  28755       580635   664819
full           1900397  55858      1282801  3362155

=== Composition: % of full corpus each stratum represents ===
                4chan    8kun  truthsocial  bluesky
query_variant                                      
israel_not      31.52   39.99        50.79    78.50
israel_and       3.85    8.53         3.95     1.73
no_israel       64.63   51.48        45.26    19.77
full           100.00  100.00       100.00   100.00

=== Composition by site ===

4chan: total = 1,900,397
  no_israel     1,228,316 (64.63%)
  israel_and       73,148 ( 3.85%)
  israel_not      598,933 (31.52%)

8kun: total = 55,858
  no_israel        28,755 (51.48%)
  israel_and        4,765 ( 8.53%)
  israel_not       22,338 

In [ ]:
import pandas as pd

df = pd.json_normalize(data["results"])
df.to_parquet("test_query_results.parquet", engine="pyarrow", index=False)
print(f"Saved {len(df)} rows to test_query_results.parquet")